# 18 — ICA Multimodel Classification Susceptibility Analysis

This notebook tests whether ICA/source-separation quality metrics help explain classification susceptibility.

For each subject, the notebook fits ICA to broadly filtered EEG epochs and computes subject-level ICA features, including:

- IC activation kurtosis
- IC spectral entropy
- pairwise dependence between IC activations
- spatial roughness/smoothness of IC maps
- mixing/unmixing matrix condition numbers
- IC map sparsity

These ICA-derived features are correlated with subject-level within-subject classification performance for:

- LDA
- Logistic Regression
- EEGNet

The goal is to test whether subjects with cleaner, more separable, more structured ICA decompositions are easier to classify.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from scipy.stats import kurtosis, pearsonr, spearmanr
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import mutual_info_score

from oddball_eeg.bids import make_epoching_config_for_dataset
from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe
from oddball_eeg.loader import load_one_subject_grouped
from oddball_eeg.stats import safe_corr, strength_word, zscore_within_group
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


set_pandas_display_options()
mne.set_log_level("ERROR")

# ------------------------------------------------------------
# Experiments
# ------------------------------------------------------------

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

MODEL_NAMES = ["lda", "logreg", "eegnet"]

MODEL_LABELS = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}

# ------------------------------------------------------------
# ICA settings
# ------------------------------------------------------------

# ICA generally benefits from high-pass filtering around 1 Hz.
ICA_L_FREQ = 1.0
ICA_H_FREQ = 45.0
ICA_NOTCH = 60.0
ICA_RESAMPLE_SFREQ = 128.0

EPOCH_TMIN = -0.2
EPOCH_TMAX = 0.8
BASELINE_WINDOW = None

ICA_N_COMPONENTS_MAX = 20
ICA_RANDOM_STATE = 97
ICA_MAX_ITER = "auto"
ICA_METHOD = "fastica"

# To speed up ICA fitting. Set to 1 if you want no decimation.
ICA_DECIM = 2

# If the ICA feature file already exists and this is False, the notebook loads it.
RECOMPUTE_ICA_FEATURES = True

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "ica_multimodel_classification_susceptibility"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

ICA_FEATURE_PATH = TABLE_DIR / "ica_subject_features.csv"

print("Output root:", OUT_ROOT)

In [ ]:
def normalize_model_name(model_value: str) -> str:
    s = str(model_value).strip().lower()

    aliases = {
        "lda": "lda",
        "linear_discriminant_analysis": "lda",
        "linear discriminant analysis": "lda",

        "logreg": "logreg",
        "logistic_regression": "logreg",
        "logistic regression": "logreg",
        "lr": "logreg",

        "eegnet": "eegnet",
        "eeg_net": "eegnet",
    }

    return aliases.get(s, s)


def normalize_subject_for_cfg(subject_value, cfg) -> str:
    s = str(subject_value).strip()

    if s.endswith(".0"):
        s = s[:-2]

    if not s.isdigit():
        return s

    configured = [str(x) for x in cfg.subjects.subjects]
    digit_lengths = [len(x) for x in configured if str(x).isdigit()]

    if len(digit_lengths) == 0:
        return s

    width = max(digit_lengths)
    return s.zfill(width)


def dataset_label_from_cfg(cfg) -> str:
    ds = cfg.dataset.dataset
    preset = cfg.dataset.event_preset

    if ds == "ds006480":
        return f"{ds}_{preset}"

    return ds


def load_subject_level_auc_for_models(exp_name: str, model_names: list[str]) -> pd.DataFrame:
    cfg = get_experiment(exp_name)

    summary_path = cfg.table_dir("summaries") / "subject_summary_across_seeds.csv"

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Missing subject summary table for {exp_name}:\n{summary_path}"
        )

    df = pd.read_csv(summary_path)

    required = {"subject", "model", "auc_seed_mean"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{summary_path} is missing required columns: {sorted(missing)}"
        )

    df = df.copy()
    df["model"] = df["model"].map(normalize_model_name)
    df["subject"] = df["subject"].map(lambda x: normalize_subject_for_cfg(x, cfg))

    df = df[df["model"].isin(model_names)].copy()

    out = pd.DataFrame({
        "experiment_name": exp_name,
        "dataset": cfg.dataset.dataset,
        "task": cfg.dataset.task,
        "event_preset": cfg.dataset.event_preset,
        "dataset_label": dataset_label_from_cfg(cfg),
        "subject": df["subject"],
        "model": df["model"],
        "model_label": df["model"].map(MODEL_LABELS),
        "auc": df["auc_seed_mean"].astype(float),
        "auc_seed_std": df.get("auc_seed_std", np.nan),
        "summary_path": str(summary_path),
    })

    return out.reset_index(drop=True)

In [ ]:
def valid_position_picks(info):
    pos = np.array([ch["loc"][:3] for ch in info["chs"]], dtype=float)
    valid = np.isfinite(pos).all(axis=1) & (np.linalg.norm(pos, axis=1) > 0)
    return np.where(valid)[0]


def _set_montage_robust(info, montage_name: str):
    info = info.copy()
    montage = mne.channels.make_standard_montage(montage_name)

    try:
        info.set_montage(
            montage,
            on_missing="ignore",
            match_case=False,
            match_alias=True,
        )
    except TypeError:
        info.set_montage(
            montage,
            on_missing="ignore",
        )

    return info


def make_eeg_info(ch_names, sfreq):
    ch_names = [str(ch) for ch in ch_names]

    info_base = mne.create_info(
        ch_names=ch_names,
        sfreq=float(sfreq),
        ch_types="eeg",
    )

    has_egi_names = any(re.fullmatch(r"E\d+", ch.upper()) for ch in ch_names)

    if has_egi_names:
        montage_candidates = [
            "GSN-HydroCel-65_1.0",
            "GSN-HydroCel-64_1.0",
            "GSN-HydroCel-128",
            "GSN-HydroCel-129",
            "standard_1020",
            "standard_1005",
            "biosemi64",
            "biosemi128",
        ]
    else:
        montage_candidates = [
            "standard_1020",
            "standard_1005",
            "biosemi64",
            "biosemi128",
            "GSN-HydroCel-65_1.0",
            "GSN-HydroCel-128",
        ]

    best_info = None
    best_name = None
    best_n_valid = -1

    for montage_name in montage_candidates:
        try:
            info_try = _set_montage_robust(info_base, montage_name)
            n_valid = len(valid_position_picks(info_try))

            if n_valid > best_n_valid:
                best_info = info_try
                best_name = montage_name
                best_n_valid = n_valid
        except Exception:
            continue

    if best_info is None:
        return info_base, "none", 0

    return best_info, best_name, best_n_valid


def hoyer_sparsity(x):
    """
    Hoyer sparsity in [0, 1], where higher means sparser.
    """
    x = np.asarray(x, dtype=float).ravel()
    x = np.abs(x)

    n = len(x)

    if n <= 1:
        return np.nan

    l1 = np.sum(x)
    l2 = np.sqrt(np.sum(x ** 2))

    if l2 <= 1e-12:
        return 0.0

    return float((np.sqrt(n) - (l1 / l2)) / (np.sqrt(n) - 1.0))


def compute_spatial_roughness_for_maps(component_maps, info, k_neighbors=4):
    """
    Estimate spatial roughness of IC maps using nearby electrode differences.

    Lower roughness means smoother spatial map.
    """
    component_maps = np.asarray(component_maps, dtype=float)

    picks = valid_position_picks(info)

    if len(picks) < 4:
        return {
            "mean_ic_spatial_roughness": np.nan,
            "median_ic_spatial_roughness": np.nan,
            "min_ic_spatial_roughness": np.nan,
            "max_ic_spatial_roughness": np.nan,
            "mean_ic_spatial_smoothness": np.nan,
        }

    pos = np.array([info["chs"][idx]["loc"][:3] for idx in picks], dtype=float)
    dist = squareform(pdist(pos))

    edges = set()

    for i in range(len(picks)):
        nearest = np.argsort(dist[i])[1:k_neighbors + 1]

        for j in nearest:
            a = int(i)
            b = int(j)

            if a != b:
                edges.add(tuple(sorted((a, b))))

    if len(edges) == 0:
        return {
            "mean_ic_spatial_roughness": np.nan,
            "median_ic_spatial_roughness": np.nan,
            "min_ic_spatial_roughness": np.nan,
            "max_ic_spatial_roughness": np.nan,
            "mean_ic_spatial_smoothness": np.nan,
        }

    edges = sorted(edges)

    roughness_values = []

    for comp_idx in range(component_maps.shape[1]):
        weights = component_maps[:, comp_idx]
        weights_valid = weights[picks]

        # Normalize map scale before roughness computation.
        sd = np.nanstd(weights_valid)

        if not np.isfinite(sd) or sd <= 1e-12:
            roughness_values.append(np.nan)
            continue

        w = (weights_valid - np.nanmean(weights_valid)) / sd

        diffs = [
            abs(w[i] - w[j])
            for i, j in edges
            if np.isfinite(w[i]) and np.isfinite(w[j])
        ]

        roughness_values.append(float(np.nanmean(diffs)) if len(diffs) else np.nan)

    roughness_values = np.asarray(roughness_values, dtype=float)

    return {
        "mean_ic_spatial_roughness": float(np.nanmean(roughness_values)),
        "median_ic_spatial_roughness": float(np.nanmedian(roughness_values)),
        "min_ic_spatial_roughness": float(np.nanmin(roughness_values)),
        "max_ic_spatial_roughness": float(np.nanmax(roughness_values)),
        "mean_ic_spatial_smoothness": float(-np.nanmean(roughness_values)),
    }

In [ ]:
def spectral_entropy_from_signal(x, sfreq, eps=1e-30):
    """
    Normalized spectral entropy for one time series.
    Higher = more broadband/noisy.
    Lower = more structured spectrum.
    """
    x = np.asarray(x, dtype=float)

    if x.size < 8:
        return np.nan

    x = x - np.nanmean(x)

    nperseg = min(int(round(float(sfreq))), len(x))

    freqs, psd = welch(
        x,
        fs=float(sfreq),
        nperseg=nperseg,
        noverlap=nperseg // 2 if nperseg >= 8 else 0,
    )

    psd = np.asarray(psd, dtype=float)
    psd = np.maximum(psd, eps)

    p = psd / np.sum(psd)

    entropy = -np.sum(p * np.log(p))
    entropy_norm = entropy / np.log(len(p))

    return float(entropy_norm)


def discretize_quantile(x, n_bins=8):
    """
    Quantile discretization for approximate mutual information.
    """
    x = np.asarray(x, dtype=float)

    if np.nanstd(x) <= 1e-12:
        return np.zeros_like(x, dtype=int)

    qs = np.linspace(0, 1, n_bins + 1)
    bins = np.nanquantile(x, qs)
    bins = np.unique(bins)

    if len(bins) <= 2:
        return np.zeros_like(x, dtype=int)

    return np.digitize(x, bins[1:-1], right=False)


def normalized_mi_discrete(a, b, eps=1e-12):
    """
    Approximate normalized mutual information for two discretized vectors.
    """
    mi = mutual_info_score(a, b)

    ha = mutual_info_score(a, a)
    hb = mutual_info_score(b, b)

    denom = np.sqrt(ha * hb) + eps

    return float(mi / denom)


def compute_activation_dependence_features(S_flat):
    """
    S_flat shape: (n_components, n_samples)
    """
    S_flat = np.asarray(S_flat, dtype=float)

    n_components = S_flat.shape[0]

    if n_components < 2:
        return {
            "mean_abs_ic_corr": np.nan,
            "median_abs_ic_corr": np.nan,
            "max_abs_ic_corr": np.nan,
            "mean_pairwise_ic_mi_approx": np.nan,
            "median_pairwise_ic_mi_approx": np.nan,
            "max_pairwise_ic_mi_approx": np.nan,
        }

    corr = np.corrcoef(S_flat)
    upper = np.triu_indices_from(corr, k=1)
    abs_corr_vals = np.abs(corr[upper])

    mi_vals = []

    discretized = [
        discretize_quantile(S_flat[i], n_bins=8)
        for i in range(n_components)
    ]

    for i, j in combinations(range(n_components), 2):
        mi_vals.append(normalized_mi_discrete(discretized[i], discretized[j]))

    mi_vals = np.asarray(mi_vals, dtype=float)

    return {
        "mean_abs_ic_corr": float(np.nanmean(abs_corr_vals)),
        "median_abs_ic_corr": float(np.nanmedian(abs_corr_vals)),
        "max_abs_ic_corr": float(np.nanmax(abs_corr_vals)),
        "mean_pairwise_ic_mi_approx": float(np.nanmean(mi_vals)),
        "median_pairwise_ic_mi_approx": float(np.nanmedian(mi_vals)),
        "max_pairwise_ic_mi_approx": float(np.nanmax(mi_vals)),
    }


def compute_ica_features_for_subject(X, ch_names, sfreq):
    """
    Fit ICA and compute source-separation quality features.

    X shape: (n_epochs, n_channels, n_times)
    """
    X = np.asarray(X, dtype=np.float64)

    n_epochs, n_channels, n_times = X.shape

    if n_channels < 4:
        raise ValueError("Too few channels for ICA.")

    info, montage_name, n_valid_pos = make_eeg_info(ch_names, sfreq)

    epochs = mne.EpochsArray(
        X,
        info,
        verbose=False,
    )

    n_components = min(
        ICA_N_COMPONENTS_MAX,
        n_channels - 1,
    )

    if n_components < 2:
        raise ValueError("Too few ICA components.")

    ica = mne.preprocessing.ICA(
        n_components=n_components,
        method=ICA_METHOD,
        random_state=ICA_RANDOM_STATE,
        max_iter=ICA_MAX_ITER,
        verbose=False,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ica.fit(
            epochs,
            decim=ICA_DECIM,
            verbose=False,
        )

    sources = ica.get_sources(epochs).get_data()
    # sources shape: (n_epochs, n_components, n_times)

    n_components_actual = sources.shape[1]
    S_flat = sources.transpose(1, 0, 2).reshape(n_components_actual, -1)

    # Activation kurtosis
    ic_kurtosis = kurtosis(
        S_flat,
        axis=1,
        fisher=True,
        bias=False,
        nan_policy="omit",
    )

    abs_ic_kurtosis = np.abs(ic_kurtosis)

    # Spectral entropy for each IC
    spectral_entropies = np.array([
        spectral_entropy_from_signal(S_flat[i], sfreq)
        for i in range(n_components_actual)
    ], dtype=float)

    # Dependence between IC activations
    dependence_features = compute_activation_dependence_features(S_flat)

    # ICA maps / mixing matrix
    component_maps = ica.get_components()
    # component_maps shape: n_channels x n_components

    map_sparsity = np.array([
        hoyer_sparsity(component_maps[:, i])
        for i in range(component_maps.shape[1])
    ], dtype=float)

    spatial_features = compute_spatial_roughness_for_maps(
        component_maps,
        info,
        k_neighbors=4,
    )

    mixing_matrix = getattr(ica, "mixing_matrix_", None)
    unmixing_matrix = getattr(ica, "unmixing_matrix_", None)

    if mixing_matrix is not None:
        mixing_condition_number = float(np.linalg.cond(mixing_matrix))
        mean_abs_mixing_weight = float(np.nanmean(np.abs(mixing_matrix)))
        std_mixing_weight = float(np.nanstd(mixing_matrix))
    else:
        mixing_condition_number = np.nan
        mean_abs_mixing_weight = np.nan
        std_mixing_weight = np.nan

    if unmixing_matrix is not None:
        unmixing_condition_number = float(np.linalg.cond(unmixing_matrix))
        mean_abs_unmixing_weight = float(np.nanmean(np.abs(unmixing_matrix)))
        std_unmixing_weight = float(np.nanstd(unmixing_matrix))
    else:
        unmixing_condition_number = np.nan
        mean_abs_unmixing_weight = np.nan
        std_unmixing_weight = np.nan

    features = {
        "n_epochs_ica": int(n_epochs),
        "n_channels_ica": int(n_channels),
        "n_times_ica": int(n_times),
        "sfreq_ica": float(sfreq),
        "n_components_ica": int(n_components_actual),
        "montage_name": montage_name,
        "n_valid_montage_positions": int(n_valid_pos),

        # Kurtosis / non-Gaussianity
        "mean_ic_kurtosis": float(np.nanmean(ic_kurtosis)),
        "median_ic_kurtosis": float(np.nanmedian(ic_kurtosis)),
        "mean_abs_ic_kurtosis": float(np.nanmean(abs_ic_kurtosis)),
        "median_abs_ic_kurtosis": float(np.nanmedian(abs_ic_kurtosis)),
        "max_abs_ic_kurtosis": float(np.nanmax(abs_ic_kurtosis)),
        "std_abs_ic_kurtosis": float(np.nanstd(abs_ic_kurtosis)),

        # Spectral entropy
        "mean_ic_spectral_entropy": float(np.nanmean(spectral_entropies)),
        "median_ic_spectral_entropy": float(np.nanmedian(spectral_entropies)),
        "min_ic_spectral_entropy": float(np.nanmin(spectral_entropies)),
        "max_ic_spectral_entropy": float(np.nanmax(spectral_entropies)),
        "std_ic_spectral_entropy": float(np.nanstd(spectral_entropies)),

        # ICA matrix stats
        "mixing_condition_number": mixing_condition_number,
        "unmixing_condition_number": unmixing_condition_number,
        "mean_abs_mixing_weight": mean_abs_mixing_weight,
        "std_mixing_weight": std_mixing_weight,
        "mean_abs_unmixing_weight": mean_abs_unmixing_weight,
        "std_unmixing_weight": std_unmixing_weight,

        # Map sparsity
        "mean_component_map_sparsity": float(np.nanmean(map_sparsity)),
        "median_component_map_sparsity": float(np.nanmedian(map_sparsity)),
        "max_component_map_sparsity": float(np.nanmax(map_sparsity)),
        "std_component_map_sparsity": float(np.nanstd(map_sparsity)),
    }

    features.update(dependence_features)
    features.update(spatial_features)

    return features

In [ ]:
auc_frames = []

for exp_name in EXPERIMENT_NAMES:
    auc_frames.append(load_subject_level_auc_for_models(exp_name, MODEL_NAMES))

multimodel_auc_df = pd.concat(auc_frames, ignore_index=True)

save_dataframe(
    multimodel_auc_df,
    TABLE_DIR / "multimodel_subject_auc.csv",
)

print("AUC rows:", len(multimodel_auc_df))
display(round_numeric_columns(multimodel_auc_df.head(20), 4))

print("\nSubjects per dataset/model:")
display(
    multimodel_auc_df
    .groupby(["dataset_label", "model"])["subject"]
    .nunique()
    .unstack()
)

In [ ]:
if ICA_FEATURE_PATH.exists() and not RECOMPUTE_ICA_FEATURES:
    print("Loading existing ICA features:", ICA_FEATURE_PATH)
    ica_feature_df = pd.read_csv(ICA_FEATURE_PATH)

else:
    print("Computing ICA features from scratch...")
    print("This may take a while.")

    feature_rows = []
    failure_rows = []

    for exp_name in EXPERIMENT_NAMES:
        cfg = get_experiment(exp_name)

        print("=" * 90)
        print("ICA feature extraction for:", exp_name)
        print("Dataset:", cfg.dataset.dataset)
        print("Task:", cfg.dataset.task)
        print("Event preset:", cfg.dataset.event_preset)
        print("=" * 90)

        ica_epoching_cfg = make_epoching_config_for_dataset(
            cfg.dataset.dataset,
            l_freq=ICA_L_FREQ,
            h_freq=ICA_H_FREQ,
            notch=ICA_NOTCH,
            resample_sfreq=ICA_RESAMPLE_SFREQ,
            tmin=EPOCH_TMIN,
            tmax=EPOCH_TMAX,
            baseline=BASELINE_WINDOW,
            event_preset=cfg.dataset.event_preset,
        )

        for subject in cfg.subjects.subjects:
            subject_norm = normalize_subject_for_cfg(subject, cfg)

            print(f"Subject {subject_norm}...")

            try:
                subject_data, times, ch_names, sfreq = load_one_subject_grouped(
                    dataset=cfg.dataset.dataset,
                    task_name=cfg.dataset.task,
                    subject=subject,
                    epoching_cfg=ica_epoching_cfg,
                    chunk_size=cfg.splits.group_chunk_size,
                    apply_baseline=False,
                    verbose=False,
                )

                features = compute_ica_features_for_subject(
                    X=subject_data.X,
                    ch_names=ch_names,
                    sfreq=sfreq,
                )

                row = {
                    "experiment_name": exp_name,
                    "dataset": cfg.dataset.dataset,
                    "task": cfg.dataset.task,
                    "event_preset": cfg.dataset.event_preset,
                    "dataset_label": dataset_label_from_cfg(cfg),
                    "subject": subject_norm,
                }

                row.update(features)

                feature_rows.append(row)

                print(
                    f"  OK: components={row['n_components_ica']}, "
                    f"kurtosis={row['mean_abs_ic_kurtosis']:.3f}, "
                    f"entropy={row['mean_ic_spectral_entropy']:.3f}"
                )

                # Save incrementally so progress is not lost.
                save_dataframe(
                    pd.DataFrame(feature_rows),
                    ICA_FEATURE_PATH,
                )

            except Exception as e:
                print(f"  FAILED: {type(e).__name__}: {e}")

                failure_rows.append({
                    "experiment_name": exp_name,
                    "dataset": cfg.dataset.dataset,
                    "task": cfg.dataset.task,
                    "event_preset": cfg.dataset.event_preset,
                    "dataset_label": dataset_label_from_cfg(cfg),
                    "subject": subject_norm,
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                })

                save_dataframe(
                    pd.DataFrame(failure_rows),
                    TABLE_DIR / "ica_feature_failures.csv",
                )

    ica_feature_df = pd.DataFrame(feature_rows)

    save_dataframe(
        ica_feature_df,
        ICA_FEATURE_PATH,
    )

print("ICA feature rows:", len(ica_feature_df))
display(round_numeric_columns(ica_feature_df.head(), 4))

In [ ]:
ica_subject_df = multimodel_auc_df.merge(
    ica_feature_df,
    on=["experiment_name", "dataset", "task", "event_preset", "dataset_label", "subject"],
    how="inner",
)

save_dataframe(
    ica_subject_df,
    TABLE_DIR / "ica_multimodel_subject_susceptibility_features.csv",
)

print("Merged rows:", len(ica_subject_df))
print("Expected AUC rows:", len(multimodel_auc_df))

display(round_numeric_columns(ica_subject_df.head(), 4))

print("\nRows per dataset/model:")
display(
    ica_subject_df
    .groupby(["dataset_label", "model"])["subject"]
    .nunique()
    .unstack()
)

In [ ]:
ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "model",
    "model_label",
    "summary_path",
    "montage_name",
}

TARGET_COLS = {
    "auc",
    "auc_seed_std",
}

def get_ica_feature_columns(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    candidates = [
        c for c in numeric_cols
        if c not in ID_COLS
           and c not in TARGET_COLS
    ]

    return candidates


ica_feature_cols = get_ica_feature_columns(ica_subject_df)

print("ICA feature columns:", len(ica_feature_cols))
for c in ica_feature_cols:
    print(" -", c)

In [ ]:
within_rows = []

for model, model_df in ica_subject_df.groupby("model"):
    for dataset_label, g in model_df.groupby("dataset_label"):
        for feature in ica_feature_cols:
            pearson_r, pearson_p, n_p = safe_corr(
                g["auc"],
                g[feature],
                method="pearson",
            )

            spearman_rho, spearman_p, n_s = safe_corr(
                g["auc"],
                g[feature],
                method="spearman",
            )

            within_rows.append({
                "model": model,
                "model_label": MODEL_LABELS.get(model, model),
                "dataset_label": dataset_label,
                "feature": feature,
                "n": n_p,
                "pearson_r": pearson_r,
                "pearson_p": pearson_p,
                "spearman_rho": spearman_rho,
                "spearman_p": spearman_p,
                "abs_pearson_r": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
                "abs_spearman_rho": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
            })

ica_within_corr_df = pd.DataFrame(within_rows)

ica_within_corr_df = ica_within_corr_df.sort_values(
    ["model", "dataset_label", "abs_spearman_rho"],
    ascending=[True, True, False],
).reset_index(drop=True)

save_dataframe(
    ica_within_corr_df,
    TABLE_DIR / "ica_within_dataset_correlations_by_model.csv",
)

display(round_numeric_columns(ica_within_corr_df.head(40), 4))

In [ ]:
z_df = ica_subject_df.copy()

z_df["auc_z"] = (
    z_df
    .groupby(["model", "dataset_label"])["auc"]
    .transform(zscore_within_group)
)

for feature in ica_feature_cols:
    z_df[f"{feature}_z"] = (
        z_df
        .groupby(["model", "dataset_label"])[feature]
        .transform(zscore_within_group)
    )

save_dataframe(
    z_df,
    TABLE_DIR / "ica_multimodel_subject_susceptibility_features_zscored.csv",
)

grand_rows = []

for model, model_df in z_df.groupby("model"):
    for feature in ica_feature_cols:
        feature_z = f"{feature}_z"

        pearson_r, pearson_p, n_p = safe_corr(
            model_df["auc_z"],
            model_df[feature_z],
            method="pearson",
        )

        spearman_rho, spearman_p, n_s = safe_corr(
            model_df["auc_z"],
            model_df[feature_z],
            method="spearman",
        )

        valid = model_df[
            np.isfinite(model_df["auc_z"])
            & np.isfinite(model_df[feature_z])
            ]

        grand_rows.append({
            "model": model,
            "model_label": MODEL_LABELS.get(model, model),
            "feature": feature,
            "n": n_p,
            "pearson_r_z": pearson_r,
            "pearson_p_z": pearson_p,
            "spearman_rho_z": spearman_rho,
            "spearman_p_z": spearman_p,
            "abs_pearson_r_z": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
            "abs_spearman_rho_z": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
            "n_valid_grand": int(len(valid)),
            "n_valid_datasets": int(valid["dataset_label"].nunique()),
            "valid_datasets": ",".join(sorted(valid["dataset_label"].unique())),
        })

ica_grand_corr_df = pd.DataFrame(grand_rows)

ica_grand_corr_df = ica_grand_corr_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(
    ica_grand_corr_df,
    TABLE_DIR / "ica_zscored_grand_correlations_by_model.csv",
)

ica_grand_corr_report_df = ica_grand_corr_df[
    (ica_grand_corr_df["n_valid_grand"] >= 40)
    & (ica_grand_corr_df["n_valid_datasets"] >= 3)
    ].copy()

ica_grand_corr_report_df = ica_grand_corr_report_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(
    ica_grand_corr_report_df,
    TABLE_DIR / "ica_zscored_grand_correlations_report_filtered_by_model.csv",
)

display(round_numeric_columns(ica_grand_corr_report_df.groupby("model").head(15), 4))

In [ ]:
ica_within_summary_df = (
    ica_within_corr_df
    .groupby(["model", "feature"])
    .agg(
        mean_within_pearson_r=("pearson_r", "mean"),
        mean_abs_within_pearson_r=("abs_pearson_r", "mean"),
        mean_within_spearman_rho=("spearman_rho", "mean"),
        mean_abs_within_spearman_rho=("abs_spearman_rho", "mean"),
        n_datasets=("dataset_label", "nunique"),
    )
    .reset_index()
)

ica_top_feature_df = ica_grand_corr_report_df.merge(
    ica_within_summary_df,
    on=["model", "feature"],
    how="left",
)

ica_top_feature_df = ica_top_feature_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(
    ica_top_feature_df,
    TABLE_DIR / "ica_top_susceptibility_features_by_model.csv",
)

display(round_numeric_columns(ica_top_feature_df.groupby("model").head(15), 4))

In [ ]:
top_n = 15

for model in MODEL_NAMES:
    model_df = ica_grand_corr_report_df[
        ica_grand_corr_report_df["model"] == model
        ].head(top_n).copy()

    if model_df.empty:
        print(f"No reportable rows for {model}")
        continue

    model_df = model_df.sort_values("spearman_rho_z", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(
        model_df["feature"],
        model_df["spearman_rho_z"],
    )

    ax.axvline(0.0, linewidth=1)
    ax.set_xlabel("Spearman correlation with z-scored AUC")
    ax.set_title(f"ICA feature correlations — {MODEL_LABELS[model]}")
    ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"ica_grand_zscored_correlation_barplot_{model}.png")
    plt.show()

In [ ]:
top_n = 20

for model in MODEL_NAMES:
    top_features = (
        ica_grand_corr_report_df[ica_grand_corr_report_df["model"] == model]
        .head(top_n)["feature"]
        .tolist()
    )

    if len(top_features) == 0:
        continue

    heat_df = ica_within_corr_df[
        (ica_within_corr_df["model"] == model)
        & (ica_within_corr_df["feature"].isin(top_features))
        ].copy()

    heat_pivot = heat_df.pivot(
        index="feature",
        columns="dataset_label",
        values="spearman_rho",
    )

    heat_pivot = heat_pivot.loc[top_features]

    fig, ax = plt.subplots(figsize=(8, max(5, 0.35 * len(top_features))))

    im = ax.imshow(
        heat_pivot.values,
        aspect="auto",
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(np.arange(len(heat_pivot.columns)))
    ax.set_xticklabels(heat_pivot.columns, rotation=45, ha="right")

    ax.set_yticks(np.arange(len(heat_pivot.index)))
    ax.set_yticklabels(heat_pivot.index)

    ax.set_title(f"ICA within-dataset correlations — {MODEL_LABELS[model]}")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("ICA feature")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Spearman rho")

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"ica_within_dataset_correlation_heatmap_{model}.png")
    plt.show()

In [ ]:
for model in MODEL_NAMES:
    model_top_features = (
        ica_grand_corr_report_df[ica_grand_corr_report_df["model"] == model]
        .head(4)["feature"]
        .tolist()
    )

    model_z_df = z_df[z_df["model"] == model].copy()

    for feature in model_top_features:
        feature_z = f"{feature}_z"

        if feature_z not in model_z_df.columns:
            continue

        row = ica_grand_corr_report_df[
            (ica_grand_corr_report_df["model"] == model)
            & (ica_grand_corr_report_df["feature"] == feature)
            ].iloc[0]

        fig, ax = plt.subplots(figsize=(7, 5))

        for dataset_label, g in model_z_df.groupby("dataset_label"):
            ax.scatter(
                g[feature_z],
                g["auc_z"],
                label=dataset_label,
                alpha=0.8,
            )

        ax.axhline(0.0, linewidth=1)
        ax.axvline(0.0, linewidth=1)

        ax.set_xlabel(f"{feature} z-scored within dataset")
        ax.set_ylabel(f"{MODEL_LABELS[model]} AUC z-scored within dataset")
        ax.set_title(
            f"ICA susceptibility — {MODEL_LABELS[model]}\n"
            f"{feature}: rho={row['spearman_rho_z']:.3f}, "
            f"p={row['spearman_p_z']:.3g}"
        )
        ax.grid(True, alpha=0.3)
        ax.legend()

        fig.tight_layout()
        save_figure(fig, FIG_DIR / f"ica_grand_z_scatter_{model}_{feature}.png")
        plt.show()

In [ ]:
ica_model_comparison_rows = []

for feature in sorted(ica_grand_corr_report_df["feature"].unique()):
    row = {"feature": feature}

    for model in MODEL_NAMES:
        sub = ica_grand_corr_report_df[
            (ica_grand_corr_report_df["model"] == model)
            & (ica_grand_corr_report_df["feature"] == feature)
            ]

        if len(sub) == 0:
            row[f"{model}_rho"] = np.nan
            row[f"{model}_p"] = np.nan
        else:
            row[f"{model}_rho"] = float(sub.iloc[0]["spearman_rho_z"])
            row[f"{model}_p"] = float(sub.iloc[0]["spearman_p_z"])

    available_rhos = [
        row.get(f"{model}_rho", np.nan)
        for model in MODEL_NAMES
    ]

    row["mean_abs_rho_across_models"] = float(np.nanmean(np.abs(available_rhos)))
    row["mean_rho_across_models"] = float(np.nanmean(available_rhos))

    ica_model_comparison_rows.append(row)

ica_model_comparison_df = pd.DataFrame(ica_model_comparison_rows)

ica_model_comparison_df = ica_model_comparison_df.sort_values(
    "mean_abs_rho_across_models",
    ascending=False,
).reset_index(drop=True)

save_dataframe(
    ica_model_comparison_df,
    TABLE_DIR / "ica_model_comparison_top_features.csv",
)

display(round_numeric_columns(ica_model_comparison_df.head(30), 4))

In [ ]:
print("=" * 90)
print("ICA multimodel susceptibility summary")
print("=" * 90)

for model in MODEL_NAMES:
    print(f"\n{MODEL_LABELS[model]} top ICA grand z-scored features:")

    model_top = ica_grand_corr_report_df[
        ica_grand_corr_report_df["model"] == model
        ].head(10)

    for _, row in model_top.iterrows():
        rho = row["spearman_rho_z"]
        p = row["spearman_p_z"]
        feature = row["feature"]

        print(
            f" - {feature}: rho={rho:.3f}, p={p:.3g}, "
            f"{strength_word(abs(rho))}"
        )

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)